In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler
from pathlib import Path
base_dir = Path().resolve().parent
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(base_dir / "data"/ "telco_customer_churn_clean.csv")
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [14]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")
df.dropna(inplace=True)

In [15]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [16]:
MonthlyChargesYes = df[df['Churn']=='Yes']['MonthlyCharges']
MonthlyChargesNo = df[df['Churn']=='No']['MonthlyCharges']

In [17]:
u_stat, p_value = mannwhitneyu(
    x = MonthlyChargesYes,
    y = MonthlyChargesNo,
    alternative = "two-sided"
)

In [18]:
print(f"U Statistic : {u_stat}\nP-Value : {p_value}")

U Statistic : 5986148.5
P-Value : 8.467195044548749e-54


In [19]:
TotalChargesYes = df[df['Churn']=='Yes']['TotalCharges']
TotalChargesNo = df[df['Churn']=='No']['TotalCharges']

In [21]:
u_stat2, p_value2 = mannwhitneyu(
    x = TotalChargesYes,
    y = TotalChargesNo,
    alternative = "two-sided"
)

In [22]:
print(f"U Statistic : {u_stat2}\nP-value : {p_value2}")

U Statistic : 3360665.0
P-value : 1.9959848938845826e-84


In [23]:
df['tenure_bins']=pd.cut(
    x = df['tenure'],
    bins = [0,12,24,36,48,60,72],
    #right=False,
    labels = ['0-12','12-24','24-36','36-48','48-60','60-72']
)

In [25]:
categorical_cols = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

vals = []
cramers_dict = {"Columns":categorical_cols,
                "Association_Strengths":vals}
for col in categorical_cols:
  interim = pd.crosstab(df[col],df['Churn'])
  v = association(interim,method="cramer")
  vals.append(v)
  print(f"Column - {col} - Done")

cramers_df = pd.DataFrame(cramers_dict)
cramers_df.sort_values(by="Association_Strengths",ascending=False)

Column - SeniorCitizen - Done
Column - Partner - Done
Column - Dependents - Done
Column - MultipleLines - Done
Column - InternetService - Done
Column - OnlineSecurity - Done
Column - OnlineBackup - Done
Column - DeviceProtection - Done
Column - TechSupport - Done
Column - StreamingTV - Done
Column - StreamingMovies - Done
Column - Contract - Done
Column - PaperlessBilling - Done
Column - PaymentMethod - Done


,Columns,Association_Strengths
11,Contract,0.409560
5,OnlineSecurity,0.346992
8,TechSupport,0.342506
4,InternetService,0.321909
13,PaymentMethod,0.302960
6,OnlineBackup,0.291902
7,DeviceProtection,0.281159
10,StreamingMovies,0.230702
9,StreamingTV,0.230143
12,PaperlessBilling,0.191454


In [26]:
nums = [[MonthlyChargesYes,MonthlyChargesNo],[TotalChargesYes,TotalChargesNo]]
for col in nums:
  all_data = np.concatenate([col[0],col[1]])
  grand_mean = np.mean(all_data)
  ss_total = np.sum((all_data - grand_mean)**2)
  ss_between = (len(col[0]) * (np.mean(col[0])-grand_mean)**2) + (len(col[1]) * (np.mean(col[1])-grand_mean)**2)
  eta_sq = ss_between/ss_total
  eta = np.sqrt(eta_sq)
  print(f"Eta is {eta}")

Eta is 0.19285821847007872
Eta is 0.19948408356756428


In [27]:
df = pd.read_csv(base_dir / "data" / "telco_customer_churn_clean.csv")
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [28]:
df.columns = ['CustomerID', 'Gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'Tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

In [30]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")
df.dropna(inplace=True)

In [31]:
df["DeviceProtection"].unique()

<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

In [32]:
data = df.drop(columns=["CustomerID"])

In [33]:
x = data.drop(columns = "Churn")
Y = data[["Churn"]]

In [36]:
numericals = []
categoricals = []
for col in x.columns:
  if x[col].dtype=="str":
    categoricals.append(col)
  else:
    numericals.append(col)

In [37]:
categoricals

['Gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [38]:
df["DeviceProtection"].unique()

<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

In [39]:
to_trans = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

for column in to_trans:
  x[column] = x[column].apply(lambda x: "No" if x=="No internet service" else x)

In [40]:
x["DeviceProtection"].unique()

<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

In [41]:
for i in x.columns:
  print()
  print(f"{i} : {x[i].nunique()}")


Gender : 2

SeniorCitizen : 2

Partner : 2

Dependents : 2

Tenure : 72

PhoneService : 2

MultipleLines : 3

InternetService : 3

OnlineSecurity : 2

OnlineBackup : 2

DeviceProtection : 2

TechSupport : 2

StreamingTV : 2

StreamingMovies : 2

Contract : 3

PaperlessBilling : 2

PaymentMethod : 4

MonthlyCharges : 1584

TotalCharges : 6530


In [42]:
x["MultipleLines"] = x["MultipleLines"].apply(lambda x: "No" if x=="No phone service" else x)

In [43]:
x["MultipleLines"].nunique()

2

In [44]:
# ohe - gender, InternetService, Contract, PaymentMethod
# oe - SeniorCitizen, Partner, Dependents, PhoneService, MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTv, StreamingMovies
ohe = OneHotEncoder(handle_unknown='ignore',
                    sparse_output=False).set_output(transform='pandas')

ohes = ["Gender", "InternetService", "Contract","PaymentMethod"]
outs = []

for column in ohes:
  lol = ohe.fit_transform(x[[column]])
  outs.append(lol)

X = outs[0]

for i in range(1,len(outs)):
  X = pd.concat([X,outs[i]],axis = 1)

oes = ["SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
outz = []

oe = OrdinalEncoder(
    handle_unknown='error',
).set_output(transform='pandas')

for column in oes:
  lolz = oe.fit_transform(x[[column]])
  outz.append(lolz)

M = outz[0]

for i in range(1,len(outz)):
  M = pd.concat([M,outz[i]],axis=1)

X = pd.concat([X,M],axis=1)

In [45]:
le = LabelEncoder()
z = le.fit_transform(Y)
y = z.reshape(z.shape[0],)
y = pd.DataFrame(
    data = y,
    columns = ["Churn"]
)
y

,Churn
0,0
1,0
2,1
3,0
4,1
...,...
7027,0
7028,0
7029,0
7030,1


In [46]:
X

,Gender_Female,Gender_Male,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),...,Partner,Dependents,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
4,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0
7039,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0
7040,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7041,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:

ss = StandardScaler().set_output(transform='pandas')
numerical_cols = ["Tenure","MonthlyCharges","TotalCharges"]
outsz = []
for column in numerical_cols:
  lolsz = ss.fit_transform(df[[column]])
  outsz.append(lolsz)
N = outsz[0]
for i in range(1,len(outsz)):
  N = pd.concat([N,outsz[i]],axis = 1)
X = pd.concat([X,N],axis=1)
X

,Gender_Female,Gender_Male,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),...,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Tenure,MonthlyCharges,TotalCharges
0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.280248,-1.161694,-0.994194
1,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.064303,-0.260878,-0.173740
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,-1.239504,-0.363923,-0.959649
3,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.512486,-0.747850,-0.195248
4,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.239504,0.196178,-0.940457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,1.0,-0.343137,0.664868,-0.129180
7039,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.612573,1.276493,2.241056
7040,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.872808,-1.170004,-0.854514
7041,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.158016,0.319168,-0.872095


In [48]:
X_cols = X.columns.to_list()
matrix = np.zeros((len(X_cols),len(X_cols)))
vif_matrix = pd.DataFrame(
    data = matrix,
    index = X_cols,
    columns = X_cols
)
vif_matrix

,Gender_Female,Gender_Male,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),...,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Tenure,MonthlyCharges,TotalCharges
Gender_Female,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gender_Male,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
InternetService_DSL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
InternetService_Fiber optic,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
InternetService_No,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Contract_Month-to-month,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Contract_One year,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Contract_Two year,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PaymentMethod_Bank transfer (automatic),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PaymentMethod_Credit card (automatic),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [49]:
for i in X_cols:
  vif_matrix.loc[i,i] = 1.0

vif_matrix

,Gender_Female,Gender_Male,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),...,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Tenure,MonthlyCharges,TotalCharges
Gender_Female,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Gender_Male,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
InternetService_DSL,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
InternetService_Fiber optic,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
InternetService_No,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Contract_Month-to-month,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Contract_One year,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Contract_Two year,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PaymentMethod_Bank transfer (automatic),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PaymentMethod_Credit card (automatic),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
X

,Gender_Female,Gender_Male,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),...,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Tenure,MonthlyCharges,TotalCharges
0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.280248,-1.161694,-0.994194
1,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.064303,-0.260878,-0.173740
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,-1.239504,-0.363923,-0.959649
3,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.512486,-0.747850,-0.195248
4,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.239504,0.196178,-0.940457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,1.0,-0.343137,0.664868,-0.129180
7039,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.612573,1.276493,2.241056
7040,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.872808,-1.170004,-0.854514
7041,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.158016,0.319168,-0.872095


In [52]:
vif_scores = {}
X = X.drop(columns=["Gender_Female","InternetService_No","Contract_Two year","PaymentMethod_Mailed check"])
X_cols = X.columns.to_list()
for feature in X_cols:
  y_vif = X[[feature]]
  X_vif = X.drop(columns=feature)
  model = LinearRegression()
  model.fit(X_vif,y_vif)
  r2 = model.score(X_vif,y_vif)
  vif = 1/(1-r2) if r2 < 1 else float("inf")
  vif_scores[feature] = vif

vif_init = pd.DataFrame(
    data = vif_scores.values(),
    columns = ["vif_score"],
    index = vif_scores.keys())

vif_init.shape

(22, 1)

In [54]:
X = X.drop(columns="MonthlyCharges")
X_cols = X.columns.to_list()

In [55]:
vif_scores = {}
for feature in X_cols:
  y_vif = X[[feature]]
  X_vif = X.drop(columns=feature)
  model = LinearRegression()
  model.fit(X_vif,y_vif)
  r2 = model.score(X_vif,y_vif)
  vif = 1/(1-r2) if r2 < 1 else float("inf")
  vif_scores[feature] = vif

vif_init = pd.DataFrame(
    data = vif_scores.values(),
    columns = ["vif_score"],
    index = vif_scores.keys())

vif_init.shape

(21, 1)

In [56]:
vif_init

,vif_score
Gender_Male,1.001888
InternetService_DSL,3.621192
InternetService_Fiber optic,4.820045
Contract_Month-to-month,3.579733
Contract_One year,1.711963
PaymentMethod_Bank transfer (automatic),1.803259
PaymentMethod_Credit card (automatic),1.773922
PaymentMethod_Electronic check,2.174103
SeniorCitizen,1.151504
Partner,1.462530


In [58]:
X = X.drop(columns="TotalCharges")
X_cols = X.columns.to_list()
vif_scores = {}
for feature in X_cols:
  y_vif = X[[feature]]
  X_vif = X.drop(columns=feature)
  model = LinearRegression()
  model.fit(X_vif,y_vif)
  r2 = model.score(X_vif,y_vif)
  vif = 1/(1-r2) if r2 < 1 else float("inf")
  vif_scores[feature] = vif

vif_init = pd.DataFrame(
    data = vif_scores.values(),
    columns = ["vif_score"],
    index = vif_scores.keys())

vif_init.shape

(20, 1)

In [59]:
vif_init

,vif_score
Gender_Male,1.001680
InternetService_DSL,3.542992
InternetService_Fiber optic,3.976396
Contract_Month-to-month,3.554048
Contract_One year,1.697233
PaymentMethod_Bank transfer (automatic),1.784717
PaymentMethod_Credit card (automatic),1.760407
PaymentMethod_Electronic check,2.132442
SeniorCitizen,1.151479
Partner,1.462363


In [60]:
vif_init.to_csv(base_dir / "data" / "vif_init_sol.csv")